# 09 — Prepare Descriptors for Complex Phase Prediction

Prepares BOP, ACE, and SOAP descriptors for all configurations of the complex TCP phases (R, M, P, δ) to be used in the prediction notebooks.

## Prerequisites / Input files
- `Fe-Mo/Atomsobjects/{R,M,P,delta}_structures.pkl` — structure objects (available on Zenodo)
- `Fe-Mo/Descriptors/PREDICTION_Fe-Mo_{R,M,P,delta}_0.7dprojections_0.5os_table_WUBIND_16.pkl` — pre-computed BOP descriptors (available on Zenodo)

## Outputs
- `Fe-Mo/Descriptors/CNAV_PREDICTION_Fe-Mo_*.pkl` — coordination-averaged prediction descriptors

## Notes
> Pre-computed BOP descriptors for the complex phases are provided in the Zenodo archive. Download them before running this notebook.

> ACE and SOAP descriptors for complex phases can be recomputed if `python-ace` and DScribe are available.



# Prepare the structures and the features for R, P, M and $\delta$ phases

In [ ]:
# --- BOP Descriptor Download Fallback ---
# The large BOP PREDICTION descriptor files (~1.3 GB total) are not stored in git.
# They are available from a Sciebo (Nextcloud) share hosted at Ruhr-Universität Bochum.
# This cell checks whether each file exists locally and downloads it if not.

import os, requests
from pathlib import Path

_SCIEBO_BASE = "https://ruhr-uni-bochum.sciebo.de/public.php/webdav"
_SCIEBO_TOKEN = "yaXRWxmTx8Et6D2"
_SCIEBO_PASSWD = "BOPDescriptorsFeMo"

_bop_prediction_files = [
    f"Fe-Mo/Descriptors/PREDICTION_Fe-Mo_{phase}_0.7dprojections_0.5os_table_WUBIND_16.pkl"
    for phase in ["R", "M", "P", "delta"]
]

for _fpath in _bop_prediction_files:
    if os.path.exists(_fpath):
        print(f"  OK (local): {_fpath}")
        continue
    _fname = os.path.basename(_fpath)
    _url = f"{_SCIEBO_BASE}/{_fname}"
    print(f"  Downloading {_fname} ...")
    Path(_fpath).parent.mkdir(parents=True, exist_ok=True)
    with requests.get(_url, auth=(_SCIEBO_TOKEN, _SCIEBO_PASSWD), stream=True) as _r:
        _r.raise_for_status()
        _total = int(_r.headers.get("content-length", 0))
        _downloaded = 0
        with open(_fpath, "wb") as _f:
            for _chunk in _r.iter_content(chunk_size=8 * 1024 * 1024):
                _f.write(_chunk)
                _downloaded += len(_chunk)
                if _total:
                    print(f"    {_downloaded / 1e6:.0f} / {_total / 1e6:.0f} MB", end="\r")
    print(f"  Done: {_fpath}")
print("BOP PREDICTION descriptors ready.")


In [ ]:
from Tools.DatasetTools.Commoms import *
from Tools.DatasetTools import GeneralFeaturizer as gf
from Tools.DatasetTools.DatasetOperator import Dataset
from Tools.PredictionTools.MakeAtomsOjects import make_all_atoms_objects, permutate, old_permutate
import joblib
try:
    from dependencies.bopdftprojections.bopdftprojections.projections import Projections
    _HAS_BOPDFTPROJ = True
except (ImportError, Exception):
    _HAS_BOPDFTPROJ = False
from Tools import PlottingTools as plotting
from Tools.DatasetTools.GeneralFeaturizer import cn_persite
from importlib.machinery import SourceFileLoader
import logging
dataset = 'Fe-Mo' #'Cr-Co-W'
NameForFile = 'FeMo'
MAG = 1
if MAG == 0:
    target_case = 'EF_nmhcp'
elif MAG == 1:
    target_case = 'EF_fmbcc'

descriptorlocation = os.path.join(dataset, 'Descriptors')
system = dataset.replace('-','')

In [ ]:
logger = logging.getLogger()
logging.basicConfig( format='%(message)s',level=logging.INFO,)


In [ ]:
import shutil

# Enable LaTeX text rendering only if a TeX Live toolchain is available.
required_tex_bins = ("latex", "dvipng", "kpsewhich")
has_texlive = all(shutil.which(cmd) is not None for cmd in required_tex_bins)

plt.rc("text", usetex=has_texlive)
plt.rc("font", family="serif", size=24)
plt.rc("xtick", labelsize=18)
plt.rc("ytick", labelsize=18)
plt.rc("axes", labelsize=18)

if not has_texlive:
    print(
        "TeX Live not fully available (missing one of: latex, dvipng, kpsewhich). "
        "Using matplotlib internal text rendering."
    )

In [ ]:
import logging

In [ ]:
from mendeleev import element


In [ ]:
DS = Dataset(dataset=dataset, remove_phases_query='Phase != "bcc" and Phase != "fcc" and Phase !="hcp"')
BS = DS.BS #pd.read_pickle(f'{dataset}/FullyCuratedParsedBriefSummary.pkl')
BS = BS.loc[~BS.index.str.contains('delta')]
TRAIN_RBS = BS.query('Phase == "R"')

In [ ]:
BS.filter(regex = "EF")

In [ ]:
target_case

In [ ]:
DS.StructureNames[DS.StructureNames == 'M']

In [ ]:
train_features = DS.Features

In [ ]:
train_features['atomic']['Structure'].max()

In [ ]:
train_features['atomic']['Structure'].min()

In [ ]:
pd.concat([ DS.StructureNames, train_features['atomic']['Structure']], axis =1 ).reset_index(drop=True).drop_duplicates().sort_values('Structure')

In [ ]:
fig, axes = plt.subplots()
axes.scatter(TRAIN_RBS['Fe_pv'], TRAIN_RBS['EF_nmhcp']*1000)
axes.set_xlabel('$x_{Fe}$')
axes.set_ylabel(r'$\Delta E_F$ (meV/at)')

In [ ]:
CNListLocation = os.path.join(descriptorlocation,'CNList.pkl')
CNList = pd.read_pickle(CNListLocation)

In [ ]:
LearningAtomsObjects = pd.read_pickle('Fe-Mo/Atomsobjects/Fe-Mo-POSCAR-initial-rescaled-AtomsObjects.pkl').query('index.str.contains("R-.*NM$")')

In [ ]:
binaries_predict = {'R' :
#                        {'permutations': LearningAtomsObjects.index.str.split('\.').map(lambda s: s[1]),
#                         'atoms_objects_file': os.path.join(dataset,'Atomsobjects/R_structures.pkl')}
                    {'permutations': permutate('R', 2, 11), 'atoms_objects_file': os.path.join(dataset,'Atomsobjects/R_structures.pkl')}, 
#                    'R_old':
#                    {'permutations': old_permutate('R', 2, 11), 'atoms_objects_file': os.path.join(dataset, 'Atomsobjects/R_structures_old.pkl')},
                    'P' : 
                    {'permutations': permutate('P', 2, 12), 'atoms_objects_file': os.path.join(dataset,'Atomsobjects/P_structures.pkl')},
#                    'P_old' : 
#                    {'permutations': old_permutate('P', 2, 12), 'atoms_objects_file': os.path.join(dataset,'Atomsobjects/P_structures.pkl')},
                    'delta': 
                    {'permutations': permutate('delta', 2, 14), 'atoms_objects_file' : os.path.join(dataset,'Atomsobjects/delta_structures.pkl') },
                    'M':
                    {'permutations': permutate('M', 2, 11 ), 'atoms_objects_file': os.path.join(dataset,'Atomsobjects', 'M_structures.pkl' )}
                   }

In [ ]:
binaries_predict['R']['permutations'][0], binaries_predict['R']['permutations'][-1], 

In [ ]:
import Tools.PredictionTools.MakeAtomsOjects as MAO

In [ ]:

atom_volumes = {'Fe': 11.734084234678496, 'Mo': 15.89162790660502}

In [ ]:
#atom_volumes = MAO.get_atom_volume_from_mp({'Fe' : 'mp-13', 'Mo' : 'mp-129'})
# was taking volumes from MP , but MPRester is not working due to incompatibilities

In [ ]:
atom_volumes

In [ ]:
for name, binary in binaries_predict.items():
    print(name)
    if os.path.exists(binary['atoms_objects_file']):
        binary['Atoms'] = pd.read_pickle(binary['atoms_objects_file'])
    else:
        binary['Atoms'] = make_all_atoms_objects(binary['permutations'], atom_volumes_def=atom_volumes)
        binary['Atoms'] = binary['Atoms'].to_frame()
        binary['Atoms'].index = binary['Atoms'].index +'.NM'
        binary['Atoms'].columns = ['atoms']
        binary['Atoms'].to_pickle(binary['atoms_objects_file'])
binaries_predict['R']['Atoms'].index = binaries_predict['R']['Atoms'].index.str.replace("-AAAAAAAAAAA", '')
binaries_predict['R']['Atoms'].index = binaries_predict['R']['Atoms'].index.str.replace("-BBBBBBBBBBB", '')

In [ ]:
binaries_predict['R']['Atoms']

In [ ]:
len(binaries_predict['R']['Atoms'].iloc[0].values[0])

In [ ]:
len(binaries_predict['M']['Atoms'].iloc[0].values[0])

In [ ]:
len(binaries_predict['P']['Atoms'].iloc[0].values[0])

In [ ]:
len(binaries_predict['delta']['Atoms'].iloc[0].values[0])

## compare atoms objects to the ones used for learning

In [ ]:
not_created = binaries_predict['R']['Atoms'].index.difference(LearningAtomsObjects.index)

intersection = LearningAtomsObjects.index.intersection(binaries_predict['R']['Atoms'].index)

created_intersection_volumes = binaries_predict['R']['Atoms'].atoms[intersection].map(lambda a: a.get_volume())

#alt_created_intersection_volumes = binaries_predict['R_old']['Atoms'].atoms[intersection].map(lambda a: a.get_volume())


intersection_volumes = LearningAtomsObjects.atoms[intersection].map(lambda a: a.get_volume())

fig, axes = plt.subplots()
axes.scatter(intersection_volumes, created_intersection_volumes)
#axes.scatter(intersection_volumes, alt_created_intersection_volumes)
axes.plot([ created_intersection_volumes.min(), created_intersection_volumes.max() ], [ created_intersection_volumes.min(), created_intersection_volumes.max() ], '-k')
axes.set_xlabel('volume of created unit cell')
axes.set_xlabel('volume of unit cell in training set')

In [ ]:
theatoms=binaries_predict['P']['Atoms'].atoms.sample(n=1).iloc[0]
plotting.plotly_atoms(theatoms)

In [ ]:
theatoms=binaries_predict['R']['Atoms'].atoms.sample(n=1).iloc[0]
plotting.plotly_atoms(theatoms)

In [ ]:
theatoms=binaries_predict['delta']['Atoms'].atoms.sample(n=1).iloc[0]
plotting.plotly_atoms(theatoms)

In [ ]:
theatoms=binaries_predict['M']['Atoms'].atoms.sample(n=1).iloc[0]
plotting.plotly_atoms(theatoms)

# Make BS 

In [ ]:
def get_nelem(a):
    return len(np.unique(a.get_chemical_symbols()))

In [ ]:
BS_predict = {}
for name, binarypredict in binaries_predict.items():
    BS_predict[name] = binarypredict['Atoms'].atoms.map(len)
    BS_predict[name].name = 'num_atoms'
    compo = pd.DataFrame.from_dict(
        binarypredict['Atoms'].atoms.map(lambda a: pd.Series(a.symbols).value_counts().to_dict()).to_dict(),
        orient = 'index'
    ).fillna(0)
    compo['Fe_pv'] = compo['Fe'] / BS_predict[name]
    compo['Mo_sv'] = 1 - compo['Fe_pv']
    compo['Mag'] = MAG #FM
    compo['nelem'] = binarypredict['Atoms'].atoms.map(get_nelem)
    BS_predict[name] = pd.concat([BS_predict[name], compo], axis = 1)

In [ ]:
BS_predict['P']

In [ ]:
BS_predict['R']['Structure'] = 4
#BS_predict['R_old']['Structure'] = 4

In [ ]:
BS_predict['P']['Structure'] = 11

In [ ]:
BS_predict['delta']['Structure'] = 11

In [ ]:
BS_predict['M']['Structure'] = 11

In [ ]:
BS.query('Phase == "R"').columns

# CALCULATE FEATURES AND APPLY CN AVERAGE. THE ATOMS OBJECTS ARE ALREADY ORDERED !

# Load BOP Features 

In [ ]:
globalmoments = 16
model_definitions = {
    '0.7dprojections_0.5os': {'model_maker_options' : {
        'element_pairs_kwargs' : {
            'bond_integral_scale': 0.7,
        },
        'atom_blocks_kwargs': {
            'onsite_levels_scale' : 0.5,
            'select_orbitals' : {'Fe': 'd', 'Mo' : 'd'}
        },
    },
    'moments' : globalmoments
    },
}
cutoff = 'table'
atoms = ['initial', 'relaxed']
retry = False

In [ ]:
P = Projections()
P.readbxmodels()
P.get_bond_chunks()
P.get_autobonds()
P.get_all_onsite_levels()
P.get_restructured_projections()

In [ ]:
def create_modelfile(acompound, target_model_filename, modelname='projections', element_pairs_kwargs={}, atom_blocks_kwargs={} ):
    print(acompound)
    if 'canonical' not in modelname :
        model_filename = P.save_abond_bx(acompound, return_filename=True,
                                         modelname=modelname, 
                                         element_pairs_kwargs=element_pairs_kwargs,
                                         atom_blocks_kwargs=atom_blocks_kwargs)
        print(model_filename)
    else:
        model_filename = C.base_canonical #f'models/W_canonical.bx'
    shutil.copy(model_filename, target_model_filename)

In [ ]:
def replace_symbols(theatoms, replacements=None):
    new_symbols = theatoms.get_chemical_symbols()
    if replacements is not None:
        for original, replacement in replacements.items():
            new_symbols = [s.replace(original, replacement) for s in new_symbols]
    new_atoms = theatoms.copy()
    new_atoms.set_chemical_symbols(new_symbols)
    return new_atoms


In [ ]:
AtomsObjects = {}
for name, binary in binaries_predict.items():
    AtomsObjects[name]  = binary['Atoms']

In [ ]:
results = {}
resultspickle = {}

In [ ]:
elements = dataset.split('-')

In [ ]:
import shutil

In [ ]:
HAS_BOPFOX = False
try:
    from BopFoxFeaturizer.Featurizer import BopfoxFeatures
    HAS_BOPFOX = True
    print('bopfoxfeaturizer import OK')
except ImportError:
    print('BopFoxFeaturizer not available — skipping BOPfox computation block')


In [ ]:
if HAS_BOPFOX:
    from Tools.DatasetTools.Commoms import *
    os.environ['PATH']+=':'+os.path.join(os.getcwd(),'dependencies/bopfox/src/')
    from BopFoxFeaturizer.Featurizer import Featurizer, BopfoxFeatures

In [ ]:
if HAS_BOPFOX:
    for (model, definition), (case, atoms_df) in product(model_definitions.items(), AtomsObjects.items()):
        if 'moments' in definition.keys():
            thismoments = definition['moments']
        else:
            thismoments = 16
        if (model, thismoments, case) in results.keys():
            continue
        create_model_options = {}
        if 'model_maker_options' in definition.keys():
            create_model_options.update(definition['model_maker_options'])
        use_elements = copy.copy(elements)
        if 'replace atoms' in definition.keys():
            for realelement, targetelement in definition['replace atoms'].items():
                use_elements = set([s.replace(realelement, targetelement) for s in use_elements])
        components = ''.join(use_elements)
        modelsfile = os.path.join('models', f'{dataset}-{components}_{model}.bx')
        if not os.path.exists(os.path.dirname(modelsfile)):
            os.makedirs(os.path.dirname(modelsfile))
        create_modelfile(use_elements,modelsfile, modelname=model, **create_model_options,   )
        if 'replace atoms' in definition.keys():
            ApplyOnAtoms = atoms_df['atoms'].apply(replace_symbols, replacements = definition['replace atoms'])
        else:
            ApplyOnAtoms = atoms_df.atoms
        print('atoms: ', case, 'model: ', model, '  cutoff: ', cutoff, ' moments:', thismoments)
        resultspickle[(model, thismoments, case)] = os.path.join(dataset, 'Descriptors', f'PREDICTION_{dataset}_{case}_{model}_{cutoff}_WUBIND_{thismoments}.pkl')
        if not os.path.exists(resultspickle[(model, thismoments, case)]):
            cwd = os.getcwd()
            BOPC = BopfoxFeatures(
                ApplyOnAtoms,modelsfile, modelname=model,
                cutoffby=cutoff, 
                binary = os.path.join(cwd, 'dependencies', 'bopfox','src', 'bopfox'),
                moments = thismoments,
                savelog=False
                )
            BOPC.featurize_dataframe(input_pickle=resultspickle, output_pickle=resultspickle, max_workers=12)
            results[(model, thismoments, case)] = BOPC.RESULTS #pd.read_pickle(resultspickle[model]) 
            results[(model, thismoments, case)].to_pickle(resultspickle[(model, thismoments, case)])
        else:
            results[(model, thismoments, case)] = pd.read_pickle(resultspickle[(model, thismoments, case)])

In [ ]:
if HAS_BOPFOX:
    modelsfile


In [ ]:
if HAS_BOPFOX:
    for (model, thismoments, case), result in results.items(): 
        if not result.index.str.contains('NM$', regex=True).all():
            result.index = result.index+'.NM'
            results[(model, thismoments, case)].to_pickle(resultspickle[(model, thismoments, case)])

In [ ]:
cn_persite['R_old'] = cn_persite['R']
cn_persite['P_old'] = cn_persite['P']

In [ ]:
cn_persite

In [ ]:
for name, struc in cn_persite.items():
    print(name, len(struc))

In [ ]:
for phase, list_of_atoms in AtomsObjects.items():
    print(phase, len(list_of_atoms))

In [ ]:
featurescnav = {}
for name, result  in results.items():
    specialcolumns =['U_bind','U_bond_atom', 'modelsfile']#, 'U_bond_atom_list'] 
    averaged_bop_file = os.path.join(descriptorlocation,'CNAV_'+os.path.basename(resultspickle[name]))

    CNList = pd.Series([cn_persite[name[-1]]]*len(result), index=result.index)

    if os.path.exists(averaged_bop_file):
        featurescnav[name] = pd.read_pickle(averaged_bop_file).astype(float)  # for some reason some values are objects
    else:
        columnstoexpand = result.columns.drop([column for column in specialcolumns if column in result.columns])
        df = gf.array_expansions(result.dropna(), columnstoexpand)
        ThisCoordination = CNList[result.index]
        print(name)
        df = gf.featurize_dataframe(df, ThisCoordination)
        shape_factors = gf.get_shape_factors(df)
        featurescnav[name] = pd.concat(
            [
                BS_predict[name[-1]][['Mag', 'Structure']].loc[df.index], 
                result[specialcolumns[:-1]].loc[df.index],
                df, shape_factors
            ],
                axis=1)
        featurescnav[name].to_pickle(averaged_bop_file)

# Load ACE features

In [ ]:
HAS_PYACE = False
try:
    from Tools.DatasetTools.ACEDescriptors import MyPyACECalculator
    from Tools.DatasetTools.ACEDescriptors import default_options_dict as default_options_dict
    from pyace import ACEBBasisSet, PyACECalculator
    HAS_PYACE = True
    print("pyace import OK")
except (ImportError, Exception) as e:
    print(f"pyace not available ({e}) — skipping ACE descriptor computation")


In [ ]:
if HAS_PYACE:
    AceConfig = copy.copy(default_options_dict)


In [ ]:
if HAS_PYACE:
    AceConfig['elements'] = dataset.split('-')

In [ ]:
resultsace = {}

In [ ]:
if HAS_PYACE:
    model = 'ACE'
    description = 'lmax=321'
    
    ACEer = MyPyACECalculator(components=components, multispace_basis_config=AceConfig)
    acer = MyPyACECalculator(components, multispace_basis_config=AceConfig)
    
    for case, atoms_df in AtomsObjects.items():
        ApplyOnAtoms = atoms_df.atoms
        print('atoms: ', case, 'model: ', 'ACE')
        resultspickle[(model, description, case)] = os.path.join(dataset, 'Descriptors', f'PREDICTION_{dataset}_{case}_{model}_{description}.pkl')
        print('atoms: ', case, 'model: ', 'ACE', resultspickle[(model, description, case)])
        if not os.path.exists(resultspickle[(model, description, case)]):
            cwd = os.getcwd()
            results[(model, description, case)] = acer.featurize_series(atoms_df.atoms)
            results[(model, description, case)].name = 'ace_projections'
            results[(model, description, case)] = results[(model, description, case)].to_frame()
            results[(model, description, case)].to_pickle(resultspickle[(model, description, case)])
        else:
            results[(model, description, case)] = pd.read_pickle(resultspickle[(model, description, case)])
            if isinstance(results[(model, description, case)], pd.core.series.Series):
                results[(model, description, case)] = results[(model, description, case)].to_frame()
                results[(model, description, case)].columns = ['ace_projections']

In [ ]:
for (model, descriptor, case), features in results.items():
#    if 'ACE' not in model:
#        continue
    print(model, descriptor, case)
    averaged_ace_file = os.path.join(descriptorlocation,'CNAV_'+os.path.basename(resultspickle[(model, descriptor, case)]))
    columnstoexpand = features.columns
    CNList = pd.Series([cn_persite[case]]*len(features), index=features.index)
    if os.path.exists(averaged_ace_file):
        featurescnav[(model, descriptor, case)] = pd.read_pickle(averaged_ace_file)
    else:
        expanded_ace = gf.array_expansions(features, ['ace_projections'])
        featurescnav[(model, descriptor, case)] = gf.featurize_dataframe(expanded_ace, CNList)
        featurescnav[(model, descriptor, case)].to_pickle(averaged_ace_file)

# load SOAP

In [ ]:
soapcases = ['specific']

In [ ]:
from dscribe.descriptors import SOAP
from mendeleev import element
import ase
from sklearn.feature_selection import VarianceThreshold

In [ ]:
SOAPFEATURES = {}
EXPANDED_SOAP = {}
AVE_SOAP = {}
variances = {}
SEL_SOAP = {}
FINAL_SOAP = {}
soap_params = dict(
    r_cut = 4,
    n_max = 5,
    l_max = 4, # f
    sigma = 0.1,
    rbf = 'gto',
    periodic = True,
)
params_str = '__'.join([f'{key}_{val}' for key, val in soap_params.items()])
soap_features_file={}

In [ ]:
EXPANDED_SOAP.keys()

In [ ]:
if HAS_BOPFOX:
    for index, atom in atoms_df['atoms'].items():
        atom.calc = None

In [ ]:
for case, atoms_df in  AtomsObjects.items():
    for soapcase in soapcases:
        if (case, soapcase) in AVE_SOAP:
          continue
        logger.info('case: %s, soap case: %s '%(case, soapcase))
        soap_features_file.update({(case,soapcase): os.path.join(dataset,'Descriptors', f'PREDICTION__prediction__{case}__{soapcase}__{params_str}.csv')})
        if os.path.exists(soap_features_file[(case, soapcase)]):
          AVE_SOAP[(case, soapcase)] = pd.read_csv(soap_features_file[(case, soapcase)], index_col = 0)
          continue
        species = [element(s).atomic_number for s in dataset.split('-')]
        thisatomsobjects = atoms_df['atoms'].map(lambda a: copy.copy(a))
        SOAPER = SOAP(species=species, **soap_params)
        SOAPFEATURES.update({( case, soapcase ): thisatomsobjects.map(SOAPER.create)}) #,pd.DataFrame(data= columns=['SOAP'])
        EXPANDED_SOAP.update({( case,soapcase ): gf.array_expansions(SOAPFEATURES[( case, soapcase )].to_frame(name='SOAP'), ['SOAP'])})
        CNList = pd.Series([cn_persite[case]]*len(EXPANDED_SOAP[(case,soapcase)]), index=EXPANDED_SOAP[( case,soapcase )].index)
        AVE_SOAP.update({(case, soapcase): gf.featurize_dataframe(EXPANDED_SOAP[( case, soapcase )], CNList)})
        AVE_SOAP[( case, soapcase )].to_csv(soap_features_file[( case, soapcase )])

In [ ]:
for case in AtomsObjects:
    for soapcase in soapcases:
        soap_features_file.update({(case,soapcase): os.path.join(dataset,'Descriptors', f'PREDICTION__{case}__{soapcase}__{params_str}.csv')})
        AVE_SOAP[( case, soapcase )].to_csv(soap_features_file[(case, soapcase)])

## update featurescnav

In [ ]:
for (case, soapcase) in AVE_SOAP:
    featurescnav[('SOAP_specific_small', soapcase, case)] = AVE_SOAP[(case, soapcase)]

In [ ]:
featurescnav.keys()

In [ ]:
for key, featurecnav in featurescnav.items():
    print(key)
    if 'Mag' not in featurecnav.columns:
        featurescnav[key] = pd.concat([ BS_predict[key[-1]]['Mag'], featurescnav[key] ], axis=1)
    if 'Structure' not in featurecnav.columns: 
        featurescnav[key] = pd.concat([ BS_predict[key[-1]]['Structure'], featurescnav[key] ], axis = 1)

In [ ]:
for key, featurecnav in featurescnav.items():
    print(key)
    if 'Mag' not in featurecnav.columns:
        print('Mag no')
    else:
        print('Mag yea')
    if 'Structure' not in featurecnav.columns: 
        print('Structure no')
    else:
        print('Structure yea')

# LOAD MODELS 

In [ ]:
regressor_file = os.path.join(dataset, 'results', f'voting_regressor_KernelRidge.pkl')

In [ ]:
from Tools.DatasetTools.MLConveniences import filter_features

In [ ]:
voting_regressor = joblib.load(regressor_file)

# Prediction ! 

In [ ]:
voting_regressor.keys()

In [ ]:
voting_regressor.keys()

In [ ]:
ModelName = 'Kernel Ridge'
featurenames = { '0.7dprojections_0.5os': '0.7dProjections 0.5OS BOP', 'ACE': 'ACE', 'SOAP_specific_small': 'SOAP_specific_small' }

In [ ]:
Prediction_Values = {}
bag_of_predictions = {}
ERR = {}
for (modelkey, modelparam, phase), featurecnav in featurescnav.items():
    logger.info("%s, %s, %s, %s, MAG = %d" %(  modelkey, modelparam, phase, featurenames[modelkey], MAG ))
    prediction_values_location = os.path.join(dataset,'results',f'PREDICTION__{phase}__{modelkey}_{modelparam}_MAG={MAG}.csv')
    combi = (ModelName, featurenames[modelkey])
    Prediction_Values[modelkey, phase] = pd.Series(voting_regressor[combi].predict(featurecnav), index=featurecnav.index, name=f'{target_case}__{modelkey}').to_frame()
    bag_of_predictions[combi] = []
    for i, (name, estimator) in enumerate(voting_regressor[combi].named_estimators_.items()):
#        this_prediction = pd.Series(estimator.predict(featurecnav), index = featurecnav.index)
#        bag_of_predictions[combi].append(this_prediction)
        Prediction_Values[modelkey, phase][f'vote_{i}']  =pd.Series(estimator.predict(featurecnav), index = featurecnav.index)
#    bag_of_predictions[combi] = pd.DataFrame(bag_of_predictions[combi]).transpose()
    Prediction_Values[modelkey, phase]['std_votes'] = Prediction_Values[modelkey, phase].filter(regex='vote_[0-9]+').std(axis=1)#bag_of_predictions[combi].std(axis=1)
    Prediction_Values[modelkey, phase].to_csv(prediction_values_location)

In [ ]:
Prediction_Values[modelkey, phase].columns

In [ ]:
fig, axes = plt.subplots()
for key, serie in Prediction_Values.items():
    sns.histplot(Prediction_Values[key][f'{target_case}__{key[0]}'], element='poly', ax = axes)
axes.set_xlabel(r'$\Delta E_f $(eV/at)')

In [ ]:
select_model = '0.7dprojections_0.5os'

In [ ]:
if (select_model, 'R') in Prediction_Values:
    coincidence = Prediction_Values[(select_model, 'R')][f'{target_case}__{select_model}'].index.intersection(TRAIN_RBS.index)

In [ ]:
if (select_model, 'R') in Prediction_Values:
    difference =TRAIN_RBS.index.difference(coincidence)

In [ ]:
if (select_model, 'R') in Prediction_Values:
    TRAIN_RBS.loc[difference]

In [ ]:
len( featurenames)

In [ ]:
featurenames

In [ ]:
BS_predict.keys()

In [ ]:
try:
    featurename_pos ={'0.7dprojections_0.5os': 0, 'ACE': 1, 'SOAP_specific_small': 2}
    phase_pos = {'R': 0, 'P': 1, 'M': 2, 'delta': 3}
    fig, axes = plt.subplots(
        len(featurenames), len(BS_predict),
        figsize = (plt.rcParams['figure.figsize'][0]*3,plt.rcParams['figure.figsize'][1]*2),
        sharey=True, sharex=True)
    for (featurename, phase), prediction  in Prediction_Values.items():
        I = featurename_pos[featurename]
        J = phase_pos[phase]
        intersection = BS_predict[phase]['Fe_pv'].index.intersection(prediction[f'{target_case}__{featurename}'].index)
        axes[I, J].scatter(
            BS_predict[phase]['Fe_pv'][intersection],
            prediction[f'{target_case}__{featurename}'][intersection],
        )
        axes[I,J].set_title(phase)
        axes[I,J].set_ylabel(featurename)
        axes[I,J].axhline(c='k')
    fig.supylabel(r'$\Delta E_F$ (meV/at)')
except (KeyError, NameError):
    pass


In [ ]:
BS_predict['R'].keys()

In [ ]:
target_case

In [ ]:
Prediction_Values.keys()

In [ ]:
Prediction_Values[('SOAP_specific_small', 'R')]

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
from scipy.stats import pearsonr, linregress

In [ ]:
try:
    featurename_pos ={'0.7dprojections_0.5os': 0, 'ACE': 1, 'SOAP_specific_small': 2}
    featurename_label ={'0.7dprojections_0.5os': 'BOP', 'ACE': 'ACE', 'SOAP_specific_small': 'SOAP' }
    phase_pos = {'R': 0}#, 'R_old': 1, 'P': 2, 'delta': 3, 'M': 4}
    fig, axes = plt.subplots(1, len(featurename_pos), figsize=(plt.rcParams['figure.figsize'][0]*len(featurename_pos), plt.rcParams['figure.figsize'][1]), sharey = True)
    for (featuregroup, I), ax in zip(featurename_pos.items(), axes):
        test_split_samples = BS.index.intersection(BS_predict['R'].index)
        targetname = f'{target_case}__{featuregroup}'
        x = Prediction_Values[(featuregroup, 'R')][targetname][test_split_samples]*1000
        y = BS.loc[test_split_samples][target_case]*1000
        reg = np.polyfit(x, y, 1)
        pol = np.poly1d(reg)
        r2 = r2_score(y, pol(x))
        rmse = mean_squared_error(y, pol(x))
        ax.scatter(x, y) 
        ax.plot([x.min(), x.max()], pol([x.min(), x.max()]), 'k')
        ax.annotate (rf'$R^2 = {r2:.3f}$\newline RMSE = {rmse:.0f} meV/at', (0.1, 0.9), xycoords='axes fraction')
        ax.set_xlabel(featurename_label[featuregroup])
    axes[0].set_ylabel('DFT')
    fig.suptitle(r'$\Delta E_F $(eV/atom) for R structures in training set, from ...')
except (KeyError, NameError):
    pass


In [ ]:
BS_predict['P'][BS_predict['P']['Fe_pv'] == 0]

In [ ]:
try:
    Prediction_Values.keys()
except (KeyError, NameError):
    pass


In [ ]:
try:
    #featurename_pos ={'0.7dprojections_0.5os': 0, 'ACE': 1}
    phase_pos = {'R': 0, 'P': 2, 'delta': 3, 'M': 4}#, 'R_old': 1}
    phase_label = {'R': 'R', 'P': 'P', 'delta': r'$\delta$', 'M': 'M'}
    bopmodel = '0.7dprojections_0.5os'
    acemodel = 'ACE'
    fig, axes = plt.subplots(
        1, len(phase_pos),
        figsize = (plt.rcParams['figure.figsize'][0]*3,plt.rcParams['figure.figsize'][1]),
        sharey=True)
    for phase, axes  in zip(phase_pos, axes):
        if phase == 'R_old':
            continue
        intersection = Prediction_Values[(bopmodel, phase)].index.intersection(Prediction_Values[(acemodel, phase)].index)
        x = Prediction_Values[(bopmodel, phase)][target_case+'__'+bopmodel][intersection]
        y = Prediction_Values[(acemodel, phase)][target_case+'__'+acemodel].loc[intersection]
        axes.scatter(x,y,edgecolor='k')
        axes.plot([x.min(), x.max()], [y.min(), y.max()], '--k', lw=5)
        axes.set_title(phase_label[phase])
        axes.axhline(c='k')
        axes.axvline(c='k')
    
    fig.supxlabel('BOP', y=-0.01)
    fig.supylabel(acemodel, x=0.07)
    fig.savefig('Fe-Mo/graphs/Fe-Mo-PredictionDifferences.pdf')
except (KeyError, NameError):
    pass

# Predicted convex hulls 

In [ ]:
from Tools.DatasetTools.Tools import Plotting, PlottingChulls

In [ ]:
from scipy.spatial import ConvexHull

In [ ]:
from matplotlib.lines import Line2D

In [ ]:
try:
    for (model, phase), prediction_values in Prediction_Values.items() :
        if (target_case, model) not in BS_predict[phase].columns:
            BS_predict[phase] = pd.concat([BS_predict[phase], prediction_values], axis = 1)
except (KeyError, NameError):
    pass


In [ ]:
BS_predict.keys()#['R'].filter(regex=target_case)

In [ ]:
phasesloc= {'R':0, 'M': 1,'P':2 ,'delta':3}
targetsloc = {target_name: i for i, target_name in enumerate(BS_predict['R'].filter(regex='EF_nmhcp').columns)}

In [ ]:
feature_labels = {'0.7dprojections_0.5os': 'BOP', 'ACE': 'ACE', 'SOAP_specific_small': 'SOAP'}

In [ ]:
phase_labels = {'delta': r'$\delta$', 'R': '$R$', 'M': '$M$', 'P': '$P$'}

In [ ]:
experimental_range = {
    'sigma': [0.4,0.55],
    'R' : [0.6,0.65],
    'mu' : [0.6, 0.55],
    'lambda': [0.655, 0.66]
}

In [ ]:
x = experimental_range['R']

In [ ]:
BS_predict[phase].filter(regex='EF_')#columns#['Fe_pv', target_name, 'nelem']

In [ ]:
P = Plotting()
inchull = {}
opo_chull = {}
CHULLS = {}

for target_name, J in targetsloc.items():
    CHULLS[target_name] = {}
    for phase, I in phasesloc.items():
        plottable = BS_predict[phase][['Fe_pv', target_name, 'nelem']].dropna(axis=0).sort_values(by=['Fe_pv', target_name])
        chull = P.get_convex_hulls(
            {phase: plottable}, ['Fe_pv'], getproperty=target_name, viewpoint=(0.0, -10)
        )
        CHULLS[target_name].update(chull)

        all_vertices_low = np.unique(np.hstack(chull[phase].simplices[chull[phase].good]))
        inchull[(phase, target_name)] = plottable.iloc[all_vertices_low].sort_values(by='Fe_pv')

print(f'Generated convex hulls for {len(targetsloc)} targets and {len(phasesloc)} phases.')

In [ ]:
ValidationDataLocation = os.path.join(dataset, 'data', 'Validation')

## save chulls to csv

In [ ]:
for (phase, pred__model), chulldata in inchull.items():
    phase_chull_location = os.path.join(ValidationDataLocation, 'inchull', phase)
    os.makedirs(phase_chull_location, exist_ok=True)
    chull_data_loc = os.path.join(phase_chull_location, f'{ pred__model }.csv')
    chulldata.to_csv(chull_data_loc)

## save uncertainties to csv

In [ ]:
try:
    for (pred__model, phase), predictionvotes in Prediction_Values.items():
        phase_chull_location = os.path.join(ValidationDataLocation, 'inchull', phase)
        os.makedirs(phase_chull_location, exist_ok=True)
        votes_data_loc = os.path.join(phase_chull_location, f'{ pred__model }:votes.csv')
        predictionvotes.to_csv(votes_data_loc)
except (KeyError, NameError):
    pass


# Recover samples in convex hull for validation 

In [ ]:
ValidationDataLocation = os.path.join(dataset, 'data', 'Validation')

In [ ]:
if not os.path.exists(ValidationDataLocation):
    os.makedirs(ValidationDataLocation)
for (phase, target_name), inchullist in inchull.items():
    group_loc = os.path.join(ValidationDataLocation, 'inchull', phase)
    if not os.path.exists(group_loc):
        os.makedirs(group_loc)
    #inchullist.to_csv(os.path.join(group_loc, 'list.csv'))
    for index, atoms in AtomsObjects[phase]['atoms'][inchullist.index].items():
        validation_atoms_location = os.path.join(group_loc, f'{index}.vasp')
        atoms.write(validation_atoms_location, direct=True, format='vasp')
        validation_structure_location = validation_atoms_location.replace('vasp', 'cfg')
        atoms.write(validation_structure_location, format='cfg')

In [ ]:
if ('M', 'EF_nmhcp__ACE') in inchull:
    display(inchull[('M', 'EF_nmhcp__ACE')])

## Plot predicted CHULL

In [ ]:
if len(targetsloc) > 0 and len(phasesloc) > 0:
    figsize = plt.rcParams['figure.figsize']
    if len(targetsloc) > 0 and len(phasesloc) > 0:
        fig, axes = plt.subplots(
            len(targetsloc), len(phasesloc),
            sharey=True, sharex=True,
            figsize=(figsize[0] * 2.5, figsize[1] * 2),
            gridspec_kw={},
        )
    
    for target_name, J in targetsloc.items():
        featurename = target_name.replace(target_case + '__', '')
        axes[J, 0].set_ylabel(feature_labels[featurename], fontsize=22)
    
        for phase, I in phasesloc.items():
            plottable = BS_predict[phase][['Fe_pv', target_name, 'nelem']].dropna(axis=0).sort_values(by=['Fe_pv', target_name])
    
            if phase == 'R':
                axes[J, I].fill_between(experimental_range[phase], -0.1, 0.7, alpha=0.5)
    
            axes[J, I].scatter(
                plottable['Fe_pv'].values, plottable[target_name].values,
                c='brown', s=10, lw=0.1, edgecolor='w'
            )
            axes[0, I].set_title(phase_labels[phase], fontsize=22)
    
            if (phase, target_name) not in inchull:
                continue
            l = axes[J, I].plot(
                inchull[(phase, target_name)]['Fe_pv'].values,
                inchull[(phase, target_name)][target_name].values,
                '-', color='brown', label='chull', linewidth=2
            )
    
            axes[J, I].axhline(c='k')
            axes[J, I].set_ylim([-0.06, 0.7])
            axes[J, I].set_xlim([0, 1])
    
    fig.subplots_adjust(wspace=0.2, hspace=0.2)
    fig.supylabel(r'$\Delta E_F$ (eV/atom)', x=0.04, fontsize=26)
    fig.supxlabel('$x_{Fe}$', fontsize=26, y=0.04)
    fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_compare_predictions.pdf')
    fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_compare_predictions.png', dpi=300)

## R chull with uncertainties

In [ ]:
try:
    featurename_pos = {'ACE':1}#{'0.7dprojections_0.5os': 0, 'ACE': 1, 'SOAP_specific_small': 2}
    featurename_label = {'0.7dprojections_0.5os': 'BOP', 'ACE': 'ACE',  'SOAP_specific_small': 'SOAP'}
    test_errors = {'0.7dprojections_0.5os': 0.018, 'ACE':0.010, 'SOAP_specific_small': 0.015}#?
    phase_pos = {'R': 0,} # 'M': 1, 'P': 2, 'delta': 3}#, 'M': 4}
    ncols = int(len(phase_pos))
    nrows = int(len(featurename_pos))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(plt.rcParams['figure.figsize'][0]*ncols*1.2,plt.rcParams['figure.figsize'][1]*nrows*1.2),
        sharey=True,
        sharex=True
        )
    for featuregroup, I in featurename_pos.items():
        targetname = f'{target_case}__{featuregroup}'
        for phase, J in phase_pos.items():
            training_samples = BS.index.str.contains(f'\\.{phase}-')
            index1 = BS_predict[phase].index
            index2 = Prediction_Values[(featuregroup, phase)].index
            intersection = index1.intersection(index2)
            targetname = f'{target_case}__{featuregroup}'
            if (phase, targetname) not in inchull:
                continue  # convex hull not available for this combination
            this_chull = inchull[(phase, targetname)].index
            axes.scatter(BS_predict[phase]['Fe_pv'][intersection], Prediction_Values[(featuregroup, phase)][targetname][intersection]*1000, label = 'Predictions', edgecolor='w')
            axes.plot(
                BS_predict[phase]['Fe_pv'][this_chull].values,
                Prediction_Values[(featuregroup, phase)][targetname][this_chull].values*1000, 
                'k',
                label = 'convex hull'
                )
            axes.fill_between(
                BS_predict[phase]['Fe_pv'][this_chull].values,
                Prediction_Values[(featuregroup, phase)][targetname][this_chull].values*1000 - test_errors[featuregroup]*1000, 
                Prediction_Values[(featuregroup, phase)][targetname][this_chull].values*1000 + test_errors[featuregroup]*1000, 
                label = 'test error',
                alpha=0.5
                )
            axes.errorbar(
                BS_predict[phase]['Fe_pv'][this_chull].values,
                Prediction_Values[(featuregroup, phase)][targetname][this_chull].values*1000, 
                yerr= Prediction_Values[(featuregroup, phase)]['std_votes'][this_chull].values*1000,
                fmt='.k',
                capsize=5,
    #            lolims=True,
                label = 'std from bag of predictions'
                )
            if len(training_samples) <0:
                continue
    #        axes.set_title(phase_label[phase], fontsize=22)
    #    axes.set_ylabel(featurename_label[featuregroup], fontsize=22)
    axes.set_xlabel('$x_{Fe}$')#, fontsize=24)
    axes.set_ylabel(rf'$\Delta E_F$ (meV / atom)', fontsize=22)
    fig.savefig(f'{dataset}/graphs/{dataset}_R_{targetname}_convexhull.pdf')
except (KeyError, NameError):
    pass


## Predicted CHULLs with uncertainties

In [ ]:
try:
    featurename_pos = {'0.7dprojections_0.5os': 0, 'ACE': 1, 'SOAP_specific_small': 2}
    featurename_label = {'0.7dprojections_0.5os': 'BOP', 'ACE': 'ACE',  'SOAP_specific_small': 'SOAP'}
    test_errors = {'0.7dprojections_0.5os': 0.018, 'ACE':0.010, 'SOAP_specific_small': 0.015}#?
    phase_pos = {'R': 0, 'M': 1, 'P': 2, 'delta': 3}#, 'M': 4}
    #fig, axes = plt.subplots(1, len(featurename_pos), figsize=(plt.rcParams['figure.figsize'][0]*2, plt.rcParams['figure.figsize'][1]), sharey = True)
    ncols = int(len(phase_pos))
    nrows = int(len(featurename_pos))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(plt.rcParams['figure.figsize'][0]*ncols,plt.rcParams['figure.figsize'][1]*nrows),
        sharey=True,
        sharex=True
        )
    for featuregroup, I in featurename_pos.items():
        targetname = f'{target_case}__{featuregroup}'
        for phase, J in phase_pos.items():
            training_samples = BS.index.str.contains(f'\\.{phase}-')
            index1 = BS_predict[phase].index
            index2 = Prediction_Values[(featuregroup, phase)].index
            intersection = index1.intersection(index2)
            targetname = f'{target_case}__{featuregroup}'
            if (phase, targetname) not in inchull:
                continue
            this_chull = inchull[(phase, targetname)].index
            axes[I,J].scatter(BS_predict[phase]['Fe_pv'][intersection], Prediction_Values[(featuregroup, phase)][targetname][intersection]*1000, label = 'Predictions', edgecolor='w')
            axes[I,J].plot(
                BS_predict[phase]['Fe_pv'][this_chull].values,
                Prediction_Values[(featuregroup, phase)][targetname][this_chull].values*1000, 
                'k',
                label = 'convex hull'
                )
            axes[I,J].fill_between(
                BS_predict[phase]['Fe_pv'][this_chull].values,
                Prediction_Values[(featuregroup, phase)][targetname][this_chull].values*1000 - test_errors[featuregroup]*1000, 
                Prediction_Values[(featuregroup, phase)][targetname][this_chull].values*1000 + test_errors[featuregroup]*1000, 
                label = 'test error',
                alpha=0.5
                )
            axes[I,J].errorbar(
                BS_predict[phase]['Fe_pv'][this_chull].values,
                Prediction_Values[(featuregroup, phase)][targetname][this_chull].values*1000, 
                yerr= Prediction_Values[(featuregroup, phase)]['std_votes'][this_chull].values*1000,
                fmt='.k',
                capsize=5,
    #            lolims=True,
                label = 'std from bag of predictions'
                )
            if len(training_samples) <0:
                continue
    #        axes[I,J].scatter(
    #            BS['Fe_pv'][training_samples],
    #            BS[target_case][training_samples]*1000,
    #            edgecolor='k',
    #            label='DFT calculations',
    #            s = 100
    #            )
            axes[0, J].set_title(phase_label[phase], fontsize=22)
    #    axes[0,I].legend( fontsize=plt.rcParams['font.size']*0.6,bbox_to_anchor = (1, 1),)
        axes[I,0].set_ylabel(featurename_label[featuregroup], fontsize=22)
    fig.supxlabel('$x_{Fe}$', fontsize=24, y=0.04)
    fig.subplots_adjust(hspace=0.1, wspace=0.1)
    fig.supylabel(r'$\Delta E_F$ (meV / atom)', fontsize=22, x=0.075)
    fig.savefig(f'{dataset}/graphs/{dataset}_R_{targetname}_convexhull.pdf')
    #fig.supxlabel('$x_{Fe}$', y=-0.01)
    #fig.supylabel (r'$\Delta E_F$ (meV / atom)')
    #fig.legend([Line2D([0],[0], ))])
except (KeyError, NameError):
    pass


In [ ]:
try:
    BOP_vs_ACE = (Prediction_Values[(bopmodel, 'R')][f'{target_case}__{bopmodel}'] - Prediction_Values[(acemodel, 'R')][f'{target_case}__{acemodel}']).abs()
except (KeyError, NameError):
    pass

## difference as function to distance to the hull

In [ ]:
from scipy.spatial import Delaunay

In [ ]:
if ('R', f'{target_case}__{acemodel}') in inchull:
    xp = inchull[('R',f'{target_case}__{acemodel}')]['Fe_pv']
    yp = inchull[('R',f'{target_case}__{acemodel}')][f'{target_case}__{acemodel}']

In [ ]:
import pdb

In [ ]:
def distance_to_chull(x, y,  chull_def: pd.core.series.Series, valuename : str):
    y_in_chull = np.interp(x, xp, yp,)
    return y - y_in_chull

In [ ]:
if ('R', f'{target_case}__{acemodel}') in inchull:
    targetname=f'{target_case}__ACE'
    distance_to_chull(BS_predict['R']['Fe_pv']['Fe_pv53.R.NM'], Prediction_Values[(acemodel, 'R')][targetname]['Fe_pv53.R.NM'], inchull[('R',f'{target_case}__{acemodel}')], f'{target_case}__{acemodel}' )

In [ ]:
xy = None
try:
    xy = pd.concat([BS_predict['R']['Fe_pv'], Prediction_Values[(acemodel, 'R')][targetname]], axis = 1)
except (KeyError, NameError):
    pass

In [ ]:
if xy is not None:
    if ('R', f'{target_case}__{acemodel}') in inchull:
        distances_to_chull = pd.Series([], name = 'DistanceToChull')
        for index, compound in xy.iterrows():
            distances_to_chull[index] = distance_to_chull(compound['Fe_pv'], compound[targetname], inchull[('R', f'{target_case}__{acemodel}')], targetname)

In [ ]:
if xy is not None:
    selection = xy.index.str.contains('NM')

In [ ]:
if xy is not None:
    plt.scatter(xy['Fe_pv'][selection], xy[targetname][selection], c=distances_to_chull[selection])
    plt.colorbar(label='distance to chull')

In [ ]:
if xy is not None:
    fig = plt.figure()
    axes = fig.add_axes([0.1, 0.1, 0.6, 0.9])
    ax2 = fig.add_axes([0.7, 0.1, 0.3,0.9], sharey = axes)
    ax2.set_axis_off()
    axes.scatter(distances_to_chull[selection], BOP_vs_ACE[selection])
    hist = ax2.hist(BOP_vs_ACE[selection], orientation='horizontal', bins = 50,edgecolor='k')
    axes.set_xlabel ('distance to convex hull')
    axes.set_ylabel('ACE - BOP')
    fig.savefig(f'{dataset}/graphs/{dataset}_error_vs_distance_to_chull.pdf')

In [ ]:
if xy is not None:
    fig, axes = plt.subplots()
    mapble = axes.scatter(xy['Fe_pv'], xy[targetname]*1000, c=BOP_vs_ACE, s=BOP_vs_ACE*5000, edgecolor='k')#c=distances_to_chull)
    plt.colorbar(label='ace - bop', mappable=mapble)
    axes.set_xlabel(r'$x_{Fe}$')
    axes.set_ylabel(r'$\Delta E_f$ (meV/atom)')
    axes.set_title(ModelName+' / '+targetname)

In [ ]:
Phases = {'R':'R', 'delta': r'$\delta$', 'P': 'P', 'M':'M'}

In [ ]:
fig, axes = plt.subplots(1, len(Phases), sharey=True, figsize=(plt.rcParams['figure.figsize'][0]*len(Phases), plt.rcParams['figure.figsize'][1]))
for (phase, title), axes  in zip(Phases.items(), axes):
    if (phase, 'EF_nmhcp__0.7dprojections_0.5os') not in inchull or (phase, 'EF_nmhcp__ACE') not in inchull:
        continue
    inchull_bop = inchull[(phase, 'EF_nmhcp__0.7dprojections_0.5os')].index
    inchull_ace = inchull[(phase, 'EF_nmhcp__ACE')].index
    sbop, pbop = pearsonr(Prediction_Values[('0.7dprojections_0.5os', phase)]['EF_nmhcp__0.7dprojections_0.5os'][inchull_bop], Prediction_Values[('ACE', phase)]['EF_nmhcp__ACE'][inchull_bop])
    sace, pace = pearsonr(Prediction_Values[('0.7dprojections_0.5os', phase)]['EF_nmhcp__0.7dprojections_0.5os'][inchull_ace], Prediction_Values[('ACE', phase)]['EF_nmhcp__ACE'][inchull_ace])
    sns.regplot(
        x=Prediction_Values[('0.7dprojections_0.5os', phase)]['EF_nmhcp__0.7dprojections_0.5os'][inchull_bop]
        , y=Prediction_Values[('ACE', phase)]['EF_nmhcp__ACE'][inchull_bop], label = f'BOP, $R^2 = {sbop:.3f}$',
        ax=axes
    )
    sns.regplot(
        x=Prediction_Values[('0.7dprojections_0.5os', phase)]['EF_nmhcp__0.7dprojections_0.5os'][inchull_ace], 
        y=Prediction_Values[('ACE', phase)]['EF_nmhcp__ACE'][inchull_ace], label = f'ACE, $R^2 = {sace:.3f}$', ax = axes, 
    )
    axes.set_xlabel('')
    axes.set_title(title)
fig.supxlabel ('BOP', y=-0.1)
fig.supylabel ('ACE', x=0.01)
plt.savefig(f'{dataset}/graphs/samples_in_convex_hull.pdf')

# Recover samples in convex hull for validation 

In [ ]:
ValidationDataLocation = os.path.join(dataset, 'data', 'Validation')

In [ ]:
ValidationDataLocation

In [ ]:
if inchull:
    pd.concat(inchull, axis =0).to_csv(os.path.join(ValidationDataLocation, 'inchull.csv'))

In [ ]:
if inchull:
    if not os.path.exists(ValidationDataLocation):
        os.makedirs(ValidationDataLocation)
    for (phase, target_name), inchullist in inchull.items():
        group_loc = os.path.join(ValidationDataLocation, 'inchull', phase)
        if not os.path.exists(group_loc):
            os.makedirs(group_loc)
        #inchullist.to_csv(os.path.join(group_loc, 'list.csv'))
        for index, atoms in AtomsObjects[phase]['atoms'][inchullist.index].items():
            validation_atoms_location = os.path.join(group_loc, f'{index}.vasp')
            atoms.write(validation_atoms_location, direct=True, format='vasp')
            validation_structure_location = validation_atoms_location.replace('vasp', 'cfg')
            atoms.write(validation_structure_location, format='cfg')

In [ ]:
inchull.keys()

In [ ]:
if ('delta', 'EF_nmhcp__ACE') in inchull:
    display(inchull[('delta', 'EF_nmhcp__ACE')])

In [ ]:
try:
    display(inchullist)
except NameError:
    pass

In [ ]:
try:
    display(target_name)
except NameError:
    pass

In [ ]:
if opo_chull:
    for (phase, target_name), inchullist in opo_chull.items():
        group_loc = os.path.join(ValidationDataLocation, 'opochull', phase, target_name)
        if not os.path.exists(group_loc):
            os.makedirs(group_loc)
        inchullist.to_csv(os.path.join(group_loc, 'list.csv'))
        for index, atoms in AtomsObjects[phase]['atoms'][inchullist.index].items():
            validation_atoms_location = os.path.join(group_loc, f'{index}.vasp')
            atoms.write(validation_atoms_location, format='vasp',sort=True)
    

In [ ]:
if ('R','EF_nmhcp__0.7dprojections_0.5os' ) in inchull:
    display(inchull[('R','EF_nmhcp__0.7dprojections_0.5os' )])

In [ ]:
try:
    display(AtomsObjects[phase]['atoms'][inchullist.index])
except (NameError, KeyError):
    pass